In [ ]:
from pyextremes import get_extremes
from pyextremes.plotting import plot_extremes
import pandas as pd
import matplotlib.pyplot as plt

series = pd.read_csv(
    "battery_wl.csv",
    index_col=0,
    parse_dates=True,
).squeeze()

series = (
    series
    .sort_index(ascending=True)
    .astype(float)
    .dropna()
    .loc[pd.to_datetime("1925"):]
)
series = series - (series.index.array - pd.to_datetime("1992")) / pd.to_timedelta("365.2425D") * 2.87e-3



In [ ]:
from pyextremes import EVA
import scipy.stats as sstats


subset = series.loc['1980-01-01':'1998-01-01']
# subset = series
model = EVA(subset)
model.get_extremes(method="BM", block_size="365.2425D", extremes_type='low')
model.plot_extremes()
plt.show()
model.fit_model(model='MLE', distribution=sstats.gumbel_r)


In [ ]:
summary = model.get_summary(
    return_period=[1, 2, 5, 10, 25, 50, 100, 250, 500, 1000],
    alpha=0.95,
    n_samples=1000,
)
summary

In [ ]:
model
model.plot_diagnostic(alpha=0.95)
plt.show()

In [ ]:
model.plot_diagnostic(alpha=0.95)
plt.show()

In [ ]:
from pyextremes import plot_mean_residual_life
import matplotlib.pyplot as plt

res_plot = plot_mean_residual_life(series)
plt.show(res_plot)

In [ ]:
model.plot_diagnostic(alpha=0.95)
plt.show()

In [ ]:
import numpy as np
from scipy.stats import truncweibull_min
import matplotlib.pyplot as plt

# Parameters for the original Weibull distribution (before truncation)
shape_c = 2.  # Shape parameter (often k or beta)
scale = 1.  # Scale parameter (often lambda or eta)

# Truncation limits (e.g., truncate below 0.5 and above 2.0)
# These are standardized limits relative to the scale and location
lower_trunc_std = 0.5
upper_trunc_std = 2.0

# Generate random samples
num_samples = 1000
samples = truncweibull_min.rvs(
    c=shape_c,
    a=lower_trunc_std,
    b=upper_trunc_std,
    loc=0,
    scale=scale,
    size=num_samples
)

# Calculate the Probability Density Function (PDF) for plotting
x = np.linspace(truncweibull_min.ppf(0.01, shape_c, lower_trunc_std, upper_trunc_std, scale=scale),
                truncweibull_min.ppf(0.99, shape_c, lower_trunc_std, upper_trunc_std, scale=scale), 100)
pdf = truncweibull_min.pdf(x, shape_c, lower_trunc_std, upper_trunc_std, scale=scale)

# Plotting the results
plt.plot(x, pdf, 'r-', lw=2, label='Truncated Weibull PDF')
plt.hist(samples, density=True, histtype='brand', alpha=0.6, label='Sampled data')
plt.xlabel('Value')
plt.ylabel('Density')
plt.title('2-Truncated Weibull Distribution')
plt.legend()
plt.show()


In [ ]:
%matplotlib inline


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import utide

print(utide.__version__)
with open("can1998.dtf") as f:
    lines = f.readlines()

print("".join(lines[:5]))
names = ["seconds", "year", "month", "day", "hour", "elev", "flag"]

obs = pd.read_csv(
    "can1998.dtf",
    names=names,
    skipinitialspace=True,
    delim_whitespace=True,
    na_values="9.990",
)


date_cols = ["year", "month", "day", "hour"]
index = pd.to_datetime(obs[date_cols])
obs = obs.drop(date_cols, axis=1)
obs.index = index

obs.head(5)
bad = obs["flag"] == 2
corrected = obs["flag"] == 1

obs.loc[bad, "elev"] = np.nan
obs["anomaly"] = obs["elev"] - obs["elev"].mean()
obs["anomaly"] = obs["anomaly"].interpolate()
print(f"{bad.sum()} points were flagged 'bad' and interpolated")
print(f"{corrected.sum()} points were flagged 'corrected' and left unchanged")
coef = utide.solve(
    obs.index,
    obs["anomaly"],
    lat=-25,
    method="ols",
    conf_int="MC",
    verbose=False,
)
tide = utide.reconstruct(obs.index, coef, verbose=False)
t = obs.index.to_pydatetime()

fig, (ax0, ax1, ax2) = plt.subplots(figsize=(17, 5), nrows=3, sharey=True, sharex=True)

ax0.plot(t, obs.anomaly, label="Observations", color="C0")
ax1.plot(t, tide.h, label="Prediction", color="C1")
ax2.plot(t, obs.anomaly - tide.h, label="Residual", color="C2")
fig.legend(ncol=3, loc="upper center")